In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Load raw and clean structurally (same as Phase 2)
df = pd.read_csv("../data/raw/uci-secom.csv")
y = df["Pass/Fail"]
X = df.drop(columns=["Time", "Pass/Fail"])

missing_pct = X.isnull().mean()
X = X.drop(columns=missing_pct[missing_pct > 0.5].index)
X = X.drop(columns=X.columns[X.nunique() <= 1])

# Impute + scale (fit on all data is acceptable for unsupervised exploration here)
X_imp = SimpleImputer(strategy="median").fit_transform(X)
X_scaled = StandardScaler().fit_transform(X_imp)

print("Prepared:", X_scaled.shape)

Prepared: (1567, 446)


In [2]:
# contamination = expected fraction of anomalies.
# We set it near the known failure rate (~0.066) as a starting guess.
iso = IsolationForest(
    n_estimators=300,
    contamination=0.066,
    random_state=42,
    n_jobs=-1
)

# -1 = anomaly, 1 = normal
anomaly_pred = iso.fit_predict(X_scaled)

print("Anomalies flagged:", (anomaly_pred == -1).sum())
print("Normal:", (anomaly_pred == 1).sum())

Anomalies flagged: 104
Normal: 1463


In [3]:
# anomaly_pred: -1 = anomaly, 1 = normal
# y: -1 = pass, 1 = fail   (note: opposite convention!)

# Convert to clean boolean flags to avoid sign confusion
is_anomaly = (anomaly_pred == -1)      # True if flagged anomalous
is_failure = (y.values == 1)           # True if actually failed

# Cross-tabulate
from sklearn.metrics import confusion_matrix
print("Rows = actual (fail vs pass), Cols = flagged (anomaly vs normal)")
print(pd.crosstab(
    pd.Series(is_failure, name="actual_failure"),
    pd.Series(is_anomaly, name="flagged_anomaly")
))

# Of the real failures, how many did the anomaly detector catch?
caught = (is_failure & is_anomaly).sum()
print(f"\nFailures caught by anomaly detection: {caught} / {is_failure.sum()}")
print(f"Anomaly-detection recall on failures: {caught / is_failure.sum():.3f}")

Rows = actual (fail vs pass), Cols = flagged (anomaly vs normal)
flagged_anomaly  False  True 
actual_failure               
False             1374     89
True                89     15

Failures caught by anomaly detection: 15 / 104
Anomaly-detection recall on failures: 0.144


In [4]:
df_time = pd.read_csv("../data/raw/uci-secom.csv")
df_time["Time"] = pd.to_datetime(df_time["Time"])
df_time = df_time.sort_values("Time").reset_index(drop=True)

# Split chronologically: first half = "reference", second half = "current"
mid = len(df_time) // 2
print("Time range:", df_time["Time"].min(), "to", df_time["Time"].max())
print("Reference period: rows 0 to", mid)
print("Current period: rows", mid, "to", len(df_time))

Time range: 2008-01-08 02:02:00 to 2008-12-10 18:47:00
Reference period: rows 0 to 783
Current period: rows 783 to 1567


In [5]:
from scipy.stats import ks_2samp

# Same structural cleaning so columns match Phase 2
feature_cols = [c for c in df_time.columns if c not in ["Time", "Pass/Fail"]]
ref = df_time.iloc[:mid][feature_cols]
cur = df_time.iloc[mid:][feature_cols]

results = []
for col in feature_cols:
    # Drop NaNs per-column for the test
    a = ref[col].dropna()
    b = cur[col].dropna()
    # Skip columns with too little data or no variance
    if len(a) < 30 or len(b) < 30 or a.nunique() <= 1 or b.nunique() <= 1:
        continue
    stat, pval = ks_2samp(a, b)
    results.append({"sensor": col, "ks_stat": stat, "p_value": pval})

drift = pd.DataFrame(results)

# Bonferroni-style threshold: we're running ~hundreds of tests
alpha = 0.05 / len(drift)
drift["drifted"] = drift["p_value"] < alpha

print("Sensors tested:", len(drift))
print("Sensors showing significant drift:", drift["drifted"].sum())
print("\nTop 10 most-drifted sensors:")
print(drift.sort_values("ks_stat", ascending=False).head(10).round(4).to_string(index=False))

Sensors tested: 468
Sensors showing significant drift: 165

Top 10 most-drifted sensors:
sensor  ks_stat  p_value  drifted
   109   0.4182      0.0     True
   129   0.3874      0.0     True
   428   0.3742      0.0     True
   155   0.3700      0.0     True
   103   0.3623      0.0     True
   290   0.3465      0.0     True
   542   0.3346      0.0     True
   122   0.3285      0.0     True
   125   0.2950      0.0     True
   247   0.2914      0.0     True
